# مقارنة نماذج مفتوحة عربية لخانة التوليد
فعّل T4 GPU ثم شغّل الخلايا بالترتيب. كل نموذج يُحمَّل بدقة ٤ بت ويجيب عن ٢٠ سؤالًا ثابتًا، وتُحفظ الإجابات في Drive (مجلد `dia`) لتراجعها أنت (عمياء: الأسماء مخفية في العرض) وأقيس أنا الجوانب الآلية.
**معايير سُجّلت قبل التشغيل:** (١) الجواب عربي صحيح اللغة ويتعلق بالسؤال، (٢) لا تكرار ولا انقطاع في الجواب، (٣) لا رفض بلا سبب. الحكم النهائي بقراءتك لأن التقييم الآلي لا يكفي للجودة.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/dia/gen', exist_ok=True)
print('تم ✔ Drive')

In [ ]:
%%bash
pip -q install -U transformers accelerate bitsandbytes
nvidia-smi -L
echo "تم ✔ المكتبات"

In [ ]:
QUESTIONS = ["كيف حالك اليوم؟", "ما هي عاصمة فرنسا؟", "اكتب لي قصيدة قصيرة عن البحر", "ما الفرق بين الحلقة والشرط في البرمجة؟", "من هو الخوارزمي؟", "ما رأيك في الذكاء الاصطناعي؟", "لخص لي أهمية القراءة في ثلاث جمل", "ما هي الدالة في البرمجة؟", "هل تستطيع مساعدتي في دراستي؟", "اشرح لي كيف تعمل الطائرة", "ما إعراب كلمة «الطالبُ» في جملة: جاء الطالبُ؟", "شكّل هذه الجملة تشكيلًا كاملًا: ذهب الولد إلى المدرسة", "ما الفرق بين الحديث الصحيح والحسن؟", "اكتب دالة بلغة بايثون تحسب مجموع قائمة أرقام", "ترجم إلى العربية: Knowledge is power", "اشرح لي معنى الاستعارة مع مثال", "اكتب رسالة قصيرة أعتذر فيها عن التأخر عن اجتماع", "ما سبب اختلاف الفصول الأربعة؟", "هل الحوت سمكة؟", "أعطني خطة لحفظ جزء من القرآن في شهر"]
MODELS = [["ALLaM-7B", "ALLaM-AI/ALLaM-7B-Instruct-preview"], ["Fanar-9B", "QCRI/Fanar-1-9B-Instruct"], ["Jais2-8B", "inception42/Jais-2-8B-Chat"], ["Qwen3-8B", "Qwen/Qwen3-8B"]]
print(len(QUESTIONS),'سؤالًا،',len(MODELS),'نماذج')

In [ ]:
# ٤) تشغيل نموذج واحد: غيّر INDEX من ٠ إلى ٣ وأعد تشغيل الخلية لكل نموذج (٥–١٠ دقائق لكل نموذج).
# بعض النماذج (Jais وربما ALLaM) تطلب قبول الشروط في Hugging Face: إن ظهر خطأ 401/403 أرسل لي لقطته وأخبرك بالزر.
INDEX = 0
import torch, json, time, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
name, mid = MODELS[INDEX]
out_path = f'/content/drive/MyDrive/dia/gen/{name}.json'
t0 = time.time()
try:
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(mid, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16), device_map='auto', trust_remote_code=True)
    res = []
    for q in QUESTIONS:
        msgs = [{'role':'user','content':q}]
        kw = {'enable_thinking': False} if 'Qwen3' in mid else {}
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', **kw).to(model.device)
        with torch.no_grad():
            o = model.generate(ids, max_new_tokens=220, do_sample=False, repetition_penalty=1.05)
        res.append({'q': q, 'a': tok.decode(o[0][ids.shape[1]:], skip_special_tokens=True).strip()})
    json.dump({'model': mid, 'answers': res, 'seconds': round(time.time()-t0)}, open(out_path,'w'), ensure_ascii=False, indent=1)
    print('تم ✔', name, round(time.time()-t0), 'ثانية')
    for r in res[:3]: print('س:', r['q'], '\nج:', r['a'][:300], '\n')
except Exception as e:
    print('فشل', name, type(e).__name__, str(e)[:400])
finally:
    try: del model
    except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ٥) عرض الإجابات سؤالًا سؤالًا لتحكم بنفسك (أسماء النماذج مخفية: أ، ب، ج، د)
import json, glob, random
files = sorted(glob.glob('/content/drive/MyDrive/dia/gen/*.json'))
data = [json.load(open(f)) for f in files]
labels = list('أبجد')[:len(data)]
for i, q in enumerate(QUESTIONS[:20]):
    print('='*60); print('س', i+1, ':', q)
    for l, d in zip(labels, data): print(f'[{l}]', d['answers'][i]['a'][:350].replace('\n',' '), '\n')
print('الترتيب:', dict(zip(labels, [d['model'] for d in data])), '← لا تنظر إليه قبل أن تحكم')